# 10 — EIA Form 860 Retirement Schedule

**Purpose:** Attach planned retirement dates and operating-life metadata to every
generator in `generators_with_costs.parquet`. This is what E4ST needs to schedule
exogenous retirements and compute remaining economic life.

**Inputs:**
- `data/processed/generators_with_costs.parquet`
- `data/processed/network_metadata.json`
- EIA API v2 — `electricity/operating-generator-capacity/data`

**Outputs:**
- `data/processed/generators_with_retirements.parquet` ← primary handoff
- `data/processed/retirement_summary.csv`
- `data/processed/network_metadata.json` (updated with `retirement_data` key)

**Handoff condition:** output parquet has same row count as source, `year_shutdown` null rate < 1%.

In [1]:
import sys, json, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import utils

PROCESSED = PROJECT_ROOT / "data" / "processed"

# ── Load generators_with_costs.parquet and print schema ───────────────────────
gen_df = pd.read_parquet(PROCESSED / "generators_with_costs.parquet")
print(f"generators_with_costs.parquet — {len(gen_df):,} rows × {gen_df.shape[1]} columns\n")
print("Dtypes:")
print(gen_df.dtypes.to_string())
print(f"\nTop 12 technologies:")
print(gen_df["technology"].value_counts().head(12).to_string())
print(f"\nplant_id  dtype: {gen_df['plant_id'].dtype}   sample: {gen_df['plant_id'].head(3).tolist()}")
print(f"generator_id dtype: {gen_df['generator_id'].dtype}  sample: {gen_df['generator_id'].head(3).tolist()}")

generators_with_costs.parquet — 14,354 rows × 12 columns

Dtypes:
bus_id                     int64
plant_id                     str
generator_id                 str
technology                   str
fuel_type                    str
capacity_mw              float64
heat_rate_mmbtu_mwh      float64
fuel_cost_per_mmbtu      float64
vom_per_mwh              float64
fom_per_kw_yr            float64
marginal_cost_per_mwh    float64
in_giant_component          bool

Top 12 technologies:
technology
Conventional Hydroelectric                3510
Natural Gas Fired Combustion Turbine      1940
Natural Gas Fired Combined Cycle          1772
Petroleum Liquids                         1649
Landfill Gas                               996
Solar Photovoltaic                         919
Onshore Wind Turbine                       876
Natural Gas Internal Combustion Engine     836
Natural Gas Steam Turbine                  466
Conventional Steam Coal                    409
Wood/Wood Waste Biomass            

In [2]:
# ── Load network_metadata.json ─────────────────────────────────────────────────
META_PATH = PROCESSED / "network_metadata.json"
with open(META_PATH) as f:
    metadata = json.load(f)

print("data_vintage_year    :", metadata.get("data_vintage_year"))
e4st = metadata.get("e4st_case", {})
print("n_generators_retained:", e4st.get("n_generators_retained"),
      " ← giant-component subset; match-rate denominator = full parquet count", len(gen_df))

data_vintage_year    : 2024
n_generators_retained: 14333  ← giant-component subset; match-rate denominator = full parquet count 14354


In [3]:
# ── Discover available EIA API data fields (metadata endpoint) ─────────────────
# Call the endpoint WITHOUT /data to get schema documentation.
print("Querying EIA metadata endpoint…")
meta_resp = utils.eia_get("electricity/operating-generator-capacity", {})

resp_block   = meta_resp.get("response", {})
avail_data   = resp_block.get("data",   {})   # dict: {field_name: metadata}
avail_facets = resp_block.get("facets", [])   # list or dict depending on API version
avail_freq   = resp_block.get("frequency", [])

print(f"\nAvailable DATA fields ({len(avail_data)}):")
for name, desc in sorted(avail_data.items()):
    print(f"  {name!r:48s} {desc}")

print(f"\nAvailable frequencies: {avail_freq}")

# Facets may come back as a list of dicts or as a plain dict
print(f"\nAvailable FACETS ({len(avail_facets)}):")
if isinstance(avail_facets, dict):
    for name, desc in sorted(avail_facets.items()):
        print(f"  {name!r:48s} {desc}")
else:
    for item in avail_facets:
        print(f"  {item}")

# Highlight fields relevant to operating year / planned retirement
targets = ["operat", "retir"]
print("\nFields matching 'operat' or 'retir':")
for name in avail_data:
    if any(t in name.lower() for t in targets):
        print(f"  DATA : {name!r}")

Querying EIA metadata endpoint…



Available DATA fields (12):
  'county'                                         {'alias': 'County'}
  'latitude'                                       {'alias': 'Latitude'}
  'longitude'                                      {'alias': 'Longitude'}
  'nameplate-capacity-mw'                          {'alias': 'Nameplate Capacity', 'units': 'MW'}
  'net-summer-capacity-mw'                         {'alias': 'Net Summer Capacity', 'units': 'MW'}
  'net-winter-capacity-mw'                         {'alias': 'Net Winter Capacity', 'units': 'MW'}
  'operating-year-month'                           {'alias': 'Operating Date (Year and Month)'}
  'planned-derate-summer-cap-mw'                   {'alias': 'Magnitude of Planned Derate (Summer Capacity)', 'units': 'MW'}
  'planned-derate-year-month'                      {'alias': 'Planned Derate Date (Year and Month)'}
  'planned-retirement-year-month'                  {'alias': 'Planned Retirement Date (Year and Month)'}
  'planned-uprate-summer-cap-m

In [4]:
# ── API and economic-life constants ────────────────────────────────────────────
ENDPOINT  = "electricity/operating-generator-capacity/data"
PAGE_SIZE = 5000
MAX_OFFSET = 30_000   # spec: 6 pages × 5,000

# Economic life assumptions by technology category (years).
# Sources: NREL ATB 2024 (wind, solar, storage), EIA/FERC standard useful-life
# guidance (coal, gas, nuclear, hydro). Used when no planned retirement date exists.
ECON_LIFE_YEARS: dict[str, int] = {
    "coal":    50,   # coal steam turbines; FERC standard depreciation life
    "ng_cc":   40,   # natural gas combined cycle; EIA / NREL ATB 2024
    "ng_ct":   30,   # natural gas combustion turbine / peaker
    "nuclear": 60,   # includes one 20-yr NRC license renewal (assumed)
    "hydro":   80,   # major hydro; relicensing cycles extend useful life
    "wind":    25,   # NREL ATB 2024 Moderate scenario
    "solar":   30,   # utility-scale PV; NREL ATB 2024 Moderate scenario
    "storage": 20,   # lithium-ion battery; cycle-life limited
    "other":   30,   # default for landfill gas, petroleum, geothermal, etc.
}

def _tech_category(tech: str) -> str:
    """Map EIA technology string to an ECON_LIFE_YEARS key."""
    t = str(tech).lower()
    if any(x in t for x in ["coal", "lignite", "petroleum coke", "waste coal"]):
        return "coal"
    if "combined cycle" in t:
        return "ng_cc"
    if any(x in t for x in ["combustion turbine", "gas turbine",
                              "natural gas steam", "internal combustion"]):
        return "ng_ct"
    if "nuclear" in t:
        return "nuclear"
    if any(x in t for x in ["hydro", "pumped"]):
        return "hydro"
    if "wind" in t:
        return "wind"
    if any(x in t for x in ["solar", "photovoltaic"]):
        return "solar"
    if any(x in t for x in ["storage", "battery", "flywheel"]):
        return "storage"
    return "other"

def tech_econ_life(tech: str) -> int:
    return ECON_LIFE_YEARS[_tech_category(tech)]

# Sanity-check the mapping on the parquet's top technologies
print("Technology → econ_life mapping (top 12 technologies in parquet):")
print(f"  {'Technology':<52s} life  category")
for tech in gen_df["technology"].value_counts().head(12).index:
    cat = _tech_category(tech)
    print(f"  {tech:<52s} {ECON_LIFE_YEARS[cat]:3d}   {cat}")

Technology → econ_life mapping (top 12 technologies in parquet):
  Technology                                           life  category
  Conventional Hydroelectric                            80   hydro
  Natural Gas Fired Combustion Turbine                  30   ng_ct
  Natural Gas Fired Combined Cycle                      40   ng_cc
  Petroleum Liquids                                     30   other
  Landfill Gas                                          30   other
  Solar Photovoltaic                                    30   solar
  Onshore Wind Turbine                                  25   wind
  Natural Gas Internal Combustion Engine                30   ng_ct
  Natural Gas Steam Turbine                             30   ng_ct
  Conventional Steam Coal                               50   coal
  Wood/Wood Waste Biomass                               30   other
  Hydroelectric Pumped Storage                          80   hydro


In [5]:
# ── Paginated fetch — operating generators, sorted period desc ─────────────────
# Strategy: sort by period descending so pages 1–N contain the most recent snapshot.
# With ~15,000–20,000 US operating generators, MAX_OFFSET=30,000 captures them all.
# Dedup in the next cell by (plant_id, generator_id) to keep one row per generator.
FETCH_BASE = {
    "frequency":              "monthly",
    "data[0]":                "nameplate-capacity-mw",
    "data[1]":                "operating-year-month",
    "data[2]":                "planned-retirement-year-month",   # actual API field name
    "facets[status][0]":      "OP",
    "sort[0][column]":        "period",
    "sort[0][direction]":     "desc",
    "length":                 PAGE_SIZE,
}

n_pages = MAX_OFFSET // PAGE_SIZE   # = 6 pages
print(f"Fetching up to {n_pages} pages × {PAGE_SIZE:,} = {MAX_OFFSET:,} records "
      f"(sorted period desc; will dedup to latest per generator)…")

all_records: list[dict] = []
LATEST_PERIOD = None
for page in range(n_pages):
    offset = page * PAGE_SIZE
    params = {**FETCH_BASE, "offset": offset}
    resp = utils.eia_get(ENDPOINT, params)
    records = resp.get("response", {}).get("data", [])
    if not records:
        print(f"  Page {page+1}: empty — stopping early.")
        break
    if LATEST_PERIOD is None:
        LATEST_PERIOD = str(records[0]["period"])
        print(f"  Latest period in response: {LATEST_PERIOD}")
    all_records.extend(records)
    page_period = str(records[-1]["period"])
    print(f"  Page {page+1}/{n_pages}: +{len(records):,} records  "
          f"(total: {len(all_records):,}, last period on page: {page_period})")
    if len(records) < PAGE_SIZE:
        print("  Partial page — no more records.")
        break
    if page < n_pages - 1:
        time.sleep(0.4)   # polite rate-limiting

print(f"\nTotal records fetched: {len(all_records):,}")
raw_df = pd.DataFrame(all_records)
print(f"Response columns : {raw_df.columns.tolist()}")
print(raw_df.head(2).to_string())

Fetching up to 6 pages × 5,000 = 30,000 records (sorted period desc; will dedup to latest per generator)…


  Latest period in response: 2026-02
  Page 1/6: +5,000 records  (total: 5,000, last period on page: 2026-02)


  Page 2/6: +5,000 records  (total: 10,000, last period on page: 2026-02)


  Page 3/6: +5,000 records  (total: 15,000, last period on page: 2026-02)


  Page 4/6: +5,000 records  (total: 20,000, last period on page: 2026-02)


  Page 5/6: +5,000 records  (total: 25,000, last period on page: 2026-02)


  Page 6/6: +5,000 records  (total: 30,000, last period on page: 2026-01)

Total records fetched: 30,000
Response columns : ['period', 'stateid', 'stateName', 'sector', 'sectorName', 'entityid', 'entityName', 'plantid', 'plantName', 'generatorid', 'technology', 'energy_source_code', 'energy-source-desc', 'prime_mover_code', 'balancing_authority_code', 'balancing-authority-name', 'status', 'statusDescription', 'nameplate-capacity-mw', 'operating-year-month', 'planned-retirement-year-month', 'unit', 'nameplate-capacity-mw-units']
    period stateid stateName            sector        sectorName entityid                  entityName plantid   plantName generatorid         technology energy_source_code  energy-source-desc prime_mover_code balancing_authority_code balancing-authority-name status statusDescription nameplate-capacity-mw operating-year-month planned-retirement-year-month unit nameplate-capacity-mw-units
0  2026-02      AK    Alaska  electric-utility  Electric Utility    63560  S

In [6]:
# ── Rename API keys: plantid/generatorid → plant_id/generator_id ──────────────
# Spec requirement: rename immediately after fetching, before any join.
raw_df = raw_df.rename(columns={"plantid": "plant_id", "generatorid": "generator_id"})

# ── Inspect year/retirement fields present in the response ────────────────────
CANDIDATE_YEAR_FIELDS = [
    "operating-year-month",
    "operating_year", "operating-year",
    "planned-retirement-year-month",    # actual API field name (YYYY-MM)
    "planned-retirement-year",          # fallback if API changes
    "planned_retirement_year",
]
present_fields = [c for c in CANDIDATE_YEAR_FIELDS if c in raw_df.columns]
print("Year/retirement fields present in response:", present_fields)
for col in present_fields:
    non_null = raw_df[col].dropna()
    print(f"  {col!r:45s}  null={raw_df[col].isna().mean():.1%}  "
          f"sample={non_null.head(5).tolist()}")

# ── Parse year_on_eia from operating-year-month ('YYYY-MM') ───────────────────
if "operating-year-month" in raw_df.columns:
    raw_df["year_on_eia"] = (
        raw_df["operating-year-month"]
        .astype(str)
        .str[:4]
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )
elif "operating_year" in raw_df.columns:
    raw_df["year_on_eia"] = pd.to_numeric(
        raw_df["operating_year"], errors="coerce"
    ).astype("Int64")
else:
    print("WARNING: no operating-year field found — year_on_eia will be null.")
    raw_df["year_on_eia"] = pd.NA

# ── Parse planned_retirement_year ─────────────────────────────────────────────
# API returns 'planned-retirement-year-month' in 'YYYY-MM' format; extract year.
if "planned-retirement-year-month" in raw_df.columns:
    raw_df["planned_retirement_year"] = (
        raw_df["planned-retirement-year-month"]
        .astype(str)
        .str[:4]
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )
elif "planned-retirement-year" in raw_df.columns:
    raw_df["planned_retirement_year"] = pd.to_numeric(
        raw_df["planned-retirement-year"], errors="coerce"
    ).astype("Int64")
elif "planned_retirement_year" in raw_df.columns:
    raw_df["planned_retirement_year"] = pd.to_numeric(
        raw_df["planned_retirement_year"], errors="coerce"
    ).astype("Int64")
else:
    print("WARNING: no planned-retirement field found — all dates will be imputed.")
    raw_df["planned_retirement_year"] = pd.NA

null_pry = raw_df["planned_retirement_year"].isna().mean()
null_yon = raw_df["year_on_eia"].isna().mean()
print(f"\nTotal records fetched: {len(raw_df):,}")
print(f"planned_retirement_year null rate: {null_pry:.1%}  "
      "(high expected — most generators lack a filed date)")
print(f"year_on_eia null rate            : {null_yon:.1%}")

# ── Deduplicate: keep one record per (plant_id, generator_id) ─────────────────
# Annual + period-filter should already give 1 row per generator; dedup as safety.
before = len(raw_df)
eia_uniq = (
    raw_df
    .sort_values("period", ascending=False)
    .drop_duplicates(subset=["plant_id", "generator_id"], keep="first")
    .reset_index(drop=True)
)
print(f"\nDedup: {before:,} → {len(eia_uniq):,} unique (plant_id, generator_id) pairs  "
      f"(dropped {before - len(eia_uniq):,} duplicates)")

# Keep only join-relevant columns; exclude 'technology' (parquet already has it)
JOIN_COLS = ["plant_id", "generator_id", "year_on_eia", "planned_retirement_year"]
if "stateid" in eia_uniq.columns:
    JOIN_COLS.insert(2, "stateid")   # stateid: new column absent from parquet
eia_join = eia_uniq[JOIN_COLS].copy()
print(f"\nJoin table: {len(eia_join):,} rows × {len(JOIN_COLS)} cols: {JOIN_COLS}")
print(eia_join.head(3).to_string())

Year/retirement fields present in response: ['operating-year-month', 'planned-retirement-year-month']
  'operating-year-month'                         null=0.0%  sample=['2000-12', '2010-12', '2023-02', '1963-07', '1954-02']
  'planned-retirement-year-month'                null=97.6%  sample=['2030-12', '2026-06', '2027-06', '2026-12', '2026-12']

Total records fetched: 30,000
planned_retirement_year null rate: 97.6%  (high expected — most generators lack a filed date)
year_on_eia null rate            : 0.0%

Dedup: 30,000 → 25,568 unique (plant_id, generator_id) pairs  (dropped 4,432 duplicates)

Join table: 25,568 rows × 5 cols: ['plant_id', 'generator_id', 'stateid', 'year_on_eia', 'planned_retirement_year']
  plant_id generator_id stateid  year_on_eia  planned_retirement_year
0        1            2      AK         2000                     <NA>
1        1            3      AK         2010                     <NA>
2        1          5.1      AK         2023                     <NA>

In [7]:
# ── Left join onto generators_with_costs ──────────────────────────────────────
# Left join: all 14,354 parquet rows are retained; EIA data attached where available.
merged = gen_df.merge(eia_join, on=["plant_id", "generator_id"], how="left")
assert len(merged) == len(gen_df), (
    f"Row count changed after merge: {len(merged)} ≠ {len(gen_df)} — check for duplicate keys in eia_join."
)
print(f"Merge: {len(gen_df):,} parquet rows → {len(merged):,} merged rows  ✓")

# ── Match-rate diagnostics (denominator = full parquet count, not 14,333) ──────
n_matched   = int(merged["year_on_eia"].notna().sum())
n_unmatched = int(merged["year_on_eia"].isna().sum())
match_rate  = n_matched / len(merged)
print(f"\nMatch rate: {n_matched:,} / {len(merged):,} = {match_rate:.1%}")
print(f"Unmatched (→ econ_life_default, year_on=2023): {n_unmatched:,}")

if n_unmatched > 0:
    print("\nUnmatched technology breakdown:")
    print(merged.loc[merged["year_on_eia"].isna(), "technology"].value_counts().to_string())

Merge: 14,354 parquet rows → 14,354 merged rows  ✓

Match rate: 14,332 / 14,354 = 99.8%
Unmatched (→ econ_life_default, year_on=2023): 22

Unmatched technology breakdown:
technology
Landfill Gas                              7
Onshore Wind Turbine                      5
Conventional Hydroelectric                4
Natural Gas Internal Combustion Engine    2
Batteries                                 1
Geothermal                                1
Natural Gas Steam Turbine                 1
Petroleum Liquids                         1


In [8]:
# ── Compute econ_life for every row from parquet technology ────────────────────
merged["econ_life"] = merged["technology"].apply(tech_econ_life).astype(int)

# ── Impute year_on ─────────────────────────────────────────────────────────────
# Matched rows: use year_on_eia from EIA API.
# Unmatched rows: spec says impute year_on = 2023 (new builds after EIA vintage).
merged["year_on"] = merged["year_on_eia"].where(
    merged["year_on_eia"].notna(), other=pd.array([2023] * len(merged), dtype="Int64")
).astype("Int64")

# ── Compute year_shutdown ──────────────────────────────────────────────────────
# Rule (spec): year_shutdown = max(year_on + econ_life, planned_retirement_year)
#              where planned data exists; otherwise year_shutdown = year_on + econ_life.
year_on_f  = merged["year_on"].to_numpy(dtype=float, na_value=np.nan)
econ_f     = merged["econ_life"].to_numpy(dtype=float)
pry_f      = merged["planned_retirement_year"].to_numpy(dtype=float, na_value=np.nan)

implied = year_on_f + econ_f                           # year_on + econ_life
has_pry = np.isfinite(pry_f)                           # planned date exists
shutdown = np.where(has_pry, np.maximum(implied, pry_f), implied)
merged["year_shutdown"] = pd.array(shutdown.astype("int64"), dtype="Int64")

# ── Assign retirement_source ───────────────────────────────────────────────────
merged["retirement_source"] = np.select(
    [
        merged["year_on_eia"].notna() & merged["planned_retirement_year"].notna(),
        merged["year_on_eia"].notna() & merged["planned_retirement_year"].isna(),
    ],
    ["eia860_planned", "eia860_imputed"],
    default="econ_life_default",
)

# ── operating_age_2023 ─────────────────────────────────────────────────────────
merged["operating_age_2023"] = (2023 - merged["year_on"]).astype("Int64")

print("Retirement source distribution:")
print(merged["retirement_source"].value_counts().to_string())
print(f"\nyear_shutdown null rate : {merged['year_shutdown'].isna().mean():.2%}")
print(f"year_on null rate       : {merged['year_on'].isna().mean():.2%}")
print(f"operating_age_2023 range: {merged['operating_age_2023'].min()} – "
      f"{merged['operating_age_2023'].max()} yr")

print("\nSample rows:")
SHOW = ["plant_id", "generator_id", "technology", "year_on", "year_shutdown",
        "econ_life", "retirement_source", "operating_age_2023"]
print(merged[SHOW].head(8).to_string(index=False))

Retirement source distribution:
retirement_source
eia860_imputed       13915
eia860_planned         417
econ_life_default       22

year_shutdown null rate : 0.00%
year_on null rate       : 0.00%
operating_age_2023 range: -3 – 132 yr

Sample rows:
plant_id generator_id                   technology  year_on  year_shutdown  econ_life retirement_source  operating_age_2023
    3003            4   Conventional Hydroelectric     1953           2033         80    eia860_imputed                  70
    3003            3   Conventional Hydroelectric     1953           2033         80    eia860_imputed                  70
    3003            2   Conventional Hydroelectric     1953           2033         80    eia860_imputed                  70
    3003            1   Conventional Hydroelectric     1953           2033         80    eia860_imputed                  70
    1776            1   Conventional Hydroelectric     1927           2007         80    eia860_imputed                  96
    1776

In [9]:
# ── Technology summary table (methods appendix) ────────────────────────────────
summary_rows = []
for tech, grp in merged.groupby("technology"):
    n        = len(grp)
    total_mw = float(grp["capacity_mw"].sum())
    mean_age = float((2023 - grp["year_on"]).mean())
    mean_rem = float((grp["year_shutdown"] - 2023).mean())

    n_planned = int((grp["retirement_source"] == "eia860_planned").sum())
    pct_gen_planned = n_planned / n * 100 if n > 0 else 0.0

    mw_planned = float(
        grp.loc[grp["retirement_source"] == "eia860_planned", "capacity_mw"].sum()
    )
    pct_mw_planned = mw_planned / total_mw * 100 if total_mw > 0 else 0.0

    mw_2030 = float(grp.loc[grp["year_shutdown"] <= 2030, "capacity_mw"].sum())
    mw_2035 = float(grp.loc[grp["year_shutdown"] <= 2035, "capacity_mw"].sum())
    mw_2040 = float(grp.loc[grp["year_shutdown"] <= 2040, "capacity_mw"].sum())

    # Flag technologies where >50% of MW have no planned retirement date
    flag = "FLAG >50% MW no planned date" if pct_mw_planned < 50 else ""

    summary_rows.append({
        "technology":             tech,
        "n_generators":           n,
        "total_mw":               round(total_mw, 0),
        "mean_age_yr":            round(mean_age, 1),
        "mean_remaining_life_yr": round(mean_rem, 1),
        "pct_gen_planned":        round(pct_gen_planned, 1),
        "pct_mw_planned":         round(pct_mw_planned, 1),
        "mw_retiring_by_2030":    round(mw_2030, 0),
        "mw_retiring_by_2035":    round(mw_2035, 0),
        "mw_retiring_by_2040":    round(mw_2040, 0),
        "flag":                   flag,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("total_mw", ascending=False)

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 50)
print("=== Retirement Summary by Technology ===")
print(summary_df.to_string(index=False))

flagged = summary_df[summary_df["flag"] != ""]
print(f"\n{'─'*60}")
if len(flagged) == 0:
    print("All technologies: ≥50% of MW have a planned retirement date.")
else:
    print(f"Flagged technologies ({len(flagged)}) — >50% MW without planned dates:")
    print(
        flagged[["technology", "total_mw", "pct_mw_planned"]]
        .to_string(index=False)
    )

=== Retirement Summary by Technology ===
                                 technology  n_generators  total_mw  mean_age_yr  mean_remaining_life_yr  pct_gen_planned  pct_mw_planned  mw_retiring_by_2030  mw_retiring_by_2035  mw_retiring_by_2040                         flag
           Natural Gas Fired Combined Cycle          1772  284433.0         22.0                    18.1              1.3             0.7              16878.0              32590.0              55042.0 FLAG >50% MW no planned date
                    Conventional Steam Coal           409  181235.0         45.0                     7.0             21.5            23.9              99991.0             144495.0             155438.0 FLAG >50% MW no planned date
       Natural Gas Fired Combustion Turbine          1940  145516.0         25.1                     5.5              3.7             4.2              51529.0             111231.0             123275.0 FLAG >50% MW no planned date
                                    Nuc

In [10]:
# ── Build output DataFrame ─────────────────────────────────────────────────────
# All columns from original parquet, plus stateid (if joined) and the 5 new cols.
BASE_COLS   = gen_df.columns.tolist()
NEW_COLS    = ["year_on", "year_shutdown", "econ_life",
               "retirement_source", "operating_age_2023"]
EXTRA_COLS  = ["stateid"] if "stateid" in merged.columns else []

OUT_COLS = BASE_COLS + EXTRA_COLS + NEW_COLS
OUT_COLS = list(dict.fromkeys(OUT_COLS))   # deduplicate while preserving order
out_df   = merged[OUT_COLS].copy()

print(f"Output columns ({len(OUT_COLS)}):")
for c in OUT_COLS:
    flag = " ← NEW" if c in NEW_COLS + EXTRA_COLS else ""
    print(f"  {c:<35s} {str(out_df[c].dtype):<12s}{flag}")

# ── Save generators_with_retirements.parquet ──────────────────────────────────
out_path = PROCESSED / "generators_with_retirements.parquet"
out_df.to_parquet(out_path, index=False)
print(f"\nSaved parquet: {out_path}  ({out_path.stat().st_size / 1e6:.2f} MB)")

# ── Save retirement_summary.csv ───────────────────────────────────────────────
csv_path = PROCESSED / "retirement_summary.csv"
summary_df.to_csv(csv_path, index=False)
print(f"Saved CSV    : {csv_path}")

# ── Update network_metadata.json ──────────────────────────────────────────────
metadata["retirement_data"] = {
    "timestamp":                    datetime.now(timezone.utc).isoformat(),
    "source":                       "EIA API v2 — electricity/operating-generator-capacity/data",
    "eia_period":                   LATEST_PERIOD,
    "match_rate":                   round(match_rate, 4),
    "n_generators":                 len(out_df),
    "n_matched":                    n_matched,
    "n_unmatched":                  n_unmatched,
    "planned_retirement_null_rate": round(
        float(merged["planned_retirement_year"].isna().mean()), 4
    ),
    "econ_life_defaults":           ECON_LIFE_YEARS,
}
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Updated      : {META_PATH}")

Output columns (18):
  bus_id                              int64       
  plant_id                            str         
  generator_id                        str         
  technology                          str         
  fuel_type                           str         
  capacity_mw                         float64     
  heat_rate_mmbtu_mwh                 float64     
  fuel_cost_per_mmbtu                 float64     
  vom_per_mwh                         float64     
  fom_per_kw_yr                       float64     
  marginal_cost_per_mwh               float64     
  in_giant_component                  bool        
  stateid                             str          ← NEW
  year_on                             Int64        ← NEW
  year_shutdown                       Int64        ← NEW
  econ_life                           int64        ← NEW
  retirement_source                   str          ← NEW
  operating_age_2023                  Int64        ← NEW

Saved parquet: /Users/dy

In [11]:
# ── Handoff confirmation ───────────────────────────────────────────────────────
verify = pd.read_parquet(out_path)

null_ys   = verify["year_shutdown"].isna().mean()
file_mb   = out_path.stat().st_size / 1e6
row_match = len(verify) == len(gen_df)
null_ok   = null_ys < 0.01

print("=" * 60)
print("HANDOFF CHECK")
print("=" * 60)
print(f"Output file        : {out_path}")
print(f"Row count          : {len(verify):,}  (source: {len(gen_df):,})"
      f"  {'OK' if row_match else 'MISMATCH'}")
print(f"year_shutdown nulls: {null_ys:.2%}"
      f"  (spec requires <1%)  {'OK' if null_ok else 'EXCEEDS THRESHOLD'}")
print(f"File size          : {file_mb:.2f} MB")

print("\nNew columns (non-null rates):")
for c in ["stateid"] + ["year_on", "year_shutdown", "econ_life",
                         "retirement_source", "operating_age_2023"]:
    if c in verify.columns:
        nn = verify[c].notna().mean()
        print(f"  {c:<30s} non-null: {nn:.1%}")

print("\nRetirement source summary:")
print(verify["retirement_source"].value_counts().to_string())

status = "HANDOFF CONDITION MET" if (row_match and null_ok) else "NOT MET — see issues above"
print(f"\n{'='*60}")
print(status)
print("=" * 60)

HANDOFF CHECK
Output file        : /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/generators_with_retirements.parquet
Row count          : 14,354  (source: 14,354)  OK
year_shutdown nulls: 0.00%  (spec requires <1%)  OK
File size          : 0.31 MB

New columns (non-null rates):
  stateid                        non-null: 99.8%
  year_on                        non-null: 100.0%
  year_shutdown                  non-null: 100.0%
  econ_life                      non-null: 100.0%
  retirement_source              non-null: 100.0%
  operating_age_2023             non-null: 100.0%

Retirement source summary:
retirement_source
eia860_imputed       13915
eia860_planned         417
econ_life_default       22

HANDOFF CONDITION MET
